<a href="https://colab.research.google.com/github/ankit-rathi/Quantvesting_v3/blob/main/04_quantvesting_terminal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Quantvesting

## Premium Investment Terminal — Phase 2

A notebook-first executive view over the existing Quantvesting engine. **No investment calculations live in the notebook.**

In [1]:
!pip install -q ta pyxirr matplotlib plotly ipywidgets

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 533.1/533.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 32.4 MB/s eta 0:00:00


### 1. Environment & controls

Change only `PORTFOLIO_ID` for another portfolio. Keep `EOD_RUN=False` unless this is the final end-of-day snapshot.

In [2]:
from google.colab import drive
drive.mount("/content/drive")

import sys
PROJECT = "/content/drive/My Drive/quantvesting_v3"
MARKET_DATA_DIR = PROJECT + "/market_data"
PORTFOLIO_ID = "ankit"
PORTFOLIO_DATA_DIR = PROJECT + f"/portfolio_data/{PORTFOLIO_ID}"
EOD_RUN = False

sys.path.insert(0, PROJECT + "/src")

from quantvesting import (
    Quantvesting, load_config, load_market_data, load_portfolio_data,
    create_run_id, PROSPECT_DISPLAY_COLUMNS, PORTFOLIO_DISPLAY_COLUMNS,
)

RUN_ID = create_run_id()

Mounted at /content/drive


In [3]:
config = load_config(PROJECT + "/config/strategy.yaml")
qv = Quantvesting(config)

# Optional shared-market refresh; leave False unless a new Screener XLSX arrived.
REFRESH_SCREENER = False
if REFRESH_SCREENER:
    qv.ingest_screener(MARKET_DATA_DIR)

market_data = load_market_data(MARKET_DATA_DIR)
portfolio_data = load_portfolio_data(PORTFOLIO_DATA_DIR, portfolio_id=PORTFOLIO_ID)

### 2. Run the existing engine

The notebook only orchestrates the existing prospect/portfolio/decision layers.

In [4]:
df_prospects = qv.prospects(
    market_data, portfolio_data=portfolio_data, include_portfolio=True,
    portfolio_id=PORTFOLIO_ID, run_id=RUN_ID,
)

df_portfolio, portfolio_summary = qv.portfolio(
    market_data, portfolio_data=portfolio_data, eod=EOD_RUN,
    portfolio_id=PORTFOLIO_ID, run_id=RUN_ID,
)

df_prospect_actions = qv.prospect_actions(df_prospects, top_n=10)
df_portfolio_actions = qv.portfolio_actions(df_portfolio)
df_rotation = qv.capital_rotation(df_prospects, df_portfolio)

/content/drive/My Drive/quantvesting_v3/src/quantvesting/portfolio.py:436: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dates = pd.to_datetime(dates)


### 3. Executive dashboard

The top section is designed to answer four questions quickly: **What do I own? What needs attention? Where is the remaining upside? Where is the next opportunity?**

In [5]:
qv.display_terminal(
    df_portfolio=df_portfolio,
    df_prospects=df_prospects,
    portfolio_summary=portfolio_summary,
    df_rotation=df_rotation,
    df_portfolio_actions=df_portfolio_actions,
    df_prospect_actions=df_prospect_actions,
    portfolio_id=PORTFOLIO_ID,
    run_id=RUN_ID,
    run_datetime=portfolio_summary.get("run_datetime"),
)

{'current': 15771959.0,
 'deployed': 14605407.0,
 'pnl': -3522739.0,
 'xirr': 3.48,
 'positions': 87,
 'prospects': 60,
 'core_allocation_pct': 66.31303545721542,
 'legacy_allocation_pct': 33.68696454278457,
 'top5_concentration_pct': np.float64(10.213862817778248),
 'weighted_remaining_upside_pct': 0.7118494460609449,
 'median_rrr': -0.33,
 'action_count': 0,
 'prospect_candidates': 0,
 'rotation_candidates': 0}

### 4. Interactive portfolio intelligence

Use the tables for drill-down. Sorting/filtering remains delegated to the existing Jupyter/Colab table adapter.

In [6]:
qv.display_dataframe(
    df_portfolio_actions,
    columns=[c for c in PORTFOLIO_DISPLAY_COLUMNS + ["Action", "ActionReason", "ActionEvidence"]
             if c in df_portfolio_actions.columns],
    sort_by="CurrAlloc%", ascending=False,
)

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,...,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category,Action,ActionReason,ActionEvidence
75,TMPV,0.09,-23.13,85.70,42.75,274128.0,-96238.0,319869.0,600.00,-84.91,...,4.0,-0.35,2.08,9.99,XY24,NTT,AUTO,HOLD,Thesis not yet at a configured review/target t...,Captured=-54% | Remaining upside=85.7%
68,STARHEALTH,0.24,13.26,30.02,47.26,95585.0,37280.0,318403.0,761.00,-0.12,...,173.0,0.39,2.07,37.67,XY24,NTT,INSURANCE,WAIT_FOR_EXIT_WINDOW,Legacy holding retained; review for an appropr...,Captured=28% | Remaining upside=30.0%
48,JIOFIN,-0.87,-15.59,58.83,34.08,184907.0,-58038.0,314308.0,387.00,-34.37,...,34.0,-0.31,2.04,8.98,XY24,BTT,FINANCE,HOLD,Thesis not yet at a configured review/target t...,Captured=-46% | Remaining upside=58.8%
82,VBL,-0.83,-11.72,53.66,35.66,166678.0,-41226.0,310618.0,669.42,-27.82,...,13.0,-0.25,2.02,14.36,X40N,ATH,FMCG,HOLD,Thesis not yet at a configured review/target t...,Captured=-33% | Remaining upside=53.7%
73,TCS,-0.01,-33.08,85.58,24.20,265331.0,-153234.0,310039.0,4230.71,-44.31,...,2.0,-0.58,2.01,15.62,X40,ATH,IT,HOLD,Thesis not yet at a configured review/target t...,Captured=-137% | Remaining upside=85.6%
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18,COFFEEDAY,-0.41,-50.08,171.00,35.27,96919.0,-56871.0,56678.0,80.00,NaN,...,266.0,-0.59,0.37,39.77,XR,NTT,HOTELS,WAIT_FOR_EXIT_WINDOW,Legacy holding retained; review for an appropr...,Captured=-142% | Remaining upside=171.0%
63,ROUTE,0.17,-62.03,337.88,66.28,170731.0,-82536.0,50530.0,2190.73,NaN,...,141.0,-0.48,0.33,21.74,SR,ATH,IT,WAIT_FOR_EXIT_WINDOW,Legacy holding retained; review for an appropr...,Captured=-94% | Remaining upside=337.9%
58,RAJOOENG,-0.28,-50.51,182.08,39.61,92369.0,-51770.0,50730.0,143.10,NaN,...,136.0,-0.56,0.33,8.68,AR,ATH,MISC,WAIT_FOR_EXIT_WINDOW,Legacy holding retained; review for an appropr...,Captured=-128% | Remaining upside=182.1%
21,DEN,0.62,-47.76,171.94,42.07,75874.0,-40336.0,44128.0,75.00,NaN,...,235.0,-0.53,0.29,20.02,AR,NTT,ENTERTAINMENT,WAIT_FOR_EXIT_WINDOW,Legacy holding retained; review for an appropr...,Captured=-114% | Remaining upside=171.9%


In [7]:
qv.display_dataframe(
    df_prospect_actions,
    columns=[c for c in PROSPECT_DISPLAY_COLUMNS + ["Action", "Reason", "Evidence"]
             if c in df_prospect_actions.columns],
    sort_by="CumlRnk", ascending=True,
)

,Symbol,FTT,Dev%_200,Dev%_PE,Spread%,Conviction,Cyclical,RSI_14,RSP,FTT%,...,Gained%,CumlRnk,ROE%/PE,Criteria,Strategy,Category,InFolio,Action,Reason,Evidence
0,INFY,1972.00,-14.48,-41.25,15.84,X-LC,NC,46.0,70.14,76.42,...,13.45,1.0,2.1,X40,NTT,IT,DM,BUY_CANDIDATE,High-ranked opportunity within the configured ...,CumlRnk=1
1,TCS,4230.71,-12.06,-44.31,14.30,X-LC,NC,46.0,63.54,85.54,...,15.64,2.0,3.1,X40,ATH,IT,DM,BUY_CANDIDATE,High-ranked opportunity within the configured ...,CumlRnk=2
2,TMPV,600.00,-8.47,-84.91,4.51,X-LC,DC,36.0,32.81,85.70,...,9.99,3.0,54.0,XY24,NTT,AUTO,DM,BUY_CANDIDATE,High-ranked opportunity within the configured ...,CumlRnk=3
3,HINDUNILVR,2922.00,-9.71,-42.16,6.49,X-LC,NC,37.0,22.57,44.73,...,0.00,4.0,1.0,XY25,NTT,FMCG,DM,BUY_CANDIDATE,High-ranked opportunity within the configured ...,CumlRnk=4
4,BSE,4391.73,3.37,-19.43,14.54,X-LC,NC,36.0,13.19,31.17,...,64.53,5.0,1.0,X40N,ATH,MISC,NA,BUY_CANDIDATE,High-ranked opportunity within the configured ...,CumlRnk=5
5,HCLTECH,1853.42,-3.69,-14.26,11.32,X-LC,NC,54.0,86.11,40.68,...,28.69,6.0,1.2,X40,ATH,IT,DM,BUY_CANDIDATE,High-ranked opportunity within the configured ...,CumlRnk=6
6,ITC,452.00,-15.32,-31.57,10.69,X-LC,NC,28.0,22.92,69.45,...,0.00,7.0,1.7,X40,NTT,FMCG,DM,BUY_CANDIDATE,High-ranked opportunity within the configured ...,CumlRnk=7
7,M&M,3762.88,2.89,-6.96,4.86,X-LC,DC,58.0,86.81,10.03,...,17.91,8.0,0.9,X40N,ATH,AUTO,DM,BUY_CANDIDATE,High-ranked opportunity within the configured ...,CumlRnk=8
8,RELIANCE,1952.00,-5.90,-6.74,6.58,X-LC,SC,51.0,52.08,48.60,...,4.35,9.0,0.4,XY25,BTT,REFINERIES,DM+SV,BUY_CANDIDATE,High-ranked opportunity within the configured ...,CumlRnk=9
9,NESTLEIND,1377.00,9.25,-7.48,12.11,X-LC,NC,44.0,60.42,-5.59,...,27.85,10.0,1.0,XY25,NTT,FMCG,NA,BUY_CANDIDATE,High-ranked opportunity within the configured ...,CumlRnk=10


### 5. Visual views

In [8]:
qv.display_health_chart(df_portfolio)
qv.display_upside_chart(df_portfolio, top_n=12)
qv.display_prospect_opportunities(df_prospects, top_n=12)

### 6. Capital rotation review

This is advisory only. It does not issue an automatic sell instruction.

In [9]:
if df_rotation.empty:
    print("No capital-rotation candidates at the current configured thresholds.")
else:
    display(df_rotation)

No capital-rotation candidates at the current configured thresholds.


### 7. Run / date selector

Run manifests answer **which data/configuration produced a result**. EOD history provides the stored portfolio time series. Historical per-stock snapshots are not reconstructed by this selector because the current repository does not persist those per-stock snapshots.

In [10]:
qv.display_run_history_selector(portfolio_data, current_run_id=RUN_ID)

(Dropdown(description='Run:', options=('Latest', 'run_20260815_211742_e3c340b3', 'run_20260817_092256_b2dd9c6a', 'run_20260817_172740_e82ac3b5', 'run_20260817_180402_c5824d7f'), value='Latest'),
 Dropdown(description='EOD date:', options=('Latest', '06-12-2024', '10-12-2024', '11-12-2024', '16-12-2024', '17-12-2024', '20-12-2024', '21-12-2024', '27-12-2024', '30-12-2024', '31-12-2024', '01-01-2025', '02-01-2025', '03-01-2025', '06-01-2025', '07-01-2025', '08-01-2025', '10-01-2025', '13-01-2025', '14-01-2025', '16-01-2025', '17-01-2025', '20-01-2025', '21-01-2025', '22-01-2025', '23-01-2025', '24-01-2025', '27-01-2025', '28-01-2025', '29-01-2025', '30-01-2025', '31-01-2025', '01-02-2025', '03-02-2025', '04-02-2025', '05-02-2025', '06-02-2025', '07-02-2025', '10-02-2025', '11-02-2025', '12-02-2025', '13-02-2025', '14-02-2025', '18-02-2025', '24-02-2025', '25-02-2025', '27-02-2025', '28-02-2025', '03-03-2025', '04-03-2025', '06-03-2025', '07-03-2025', '10-03-2025', '11-03-2025', '18-03-20

### 8. HNI review checklist

1. Review the executive cards.
2. Read active actions and their evidence.
3. Inspect top remaining-upside holdings.
4. Inspect top prospect opportunities.
5. Review rotation candidates.
6. Check the selected run/date before sharing the report.

The investment engine remains unchanged; this notebook is the presentation layer.